In [1]:
import pandas as pd
import numpy as np



In [2]:
df = pd.read_csv("../data/processed/cleaned_diabetes_data.csv")

C:\Users\aatif\AppData\Local\Temp\ipykernel_2704\740818745.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed/cleaned_diabetes_data.csv")


In [3]:
df.shape

(101766, 50)

In [4]:
df['readmitted_30'].value_counts()

readmitted_30
0    90409
1    11357
Name: count, dtype: int64

In [5]:
diagnosis_cols = ['diag_1', 'diag_2', 'diag_3']

df[diagnosis_cols].head(10)

,diag_1,diag_2,diag_3
0,250.83,NaN,NaN
1,276,250.01,255
2,648,250,V27
3,8,250.43,403
4,197,157,250
5,414,411,250
6,414,411,V45
7,428,492,250
8,398,427,38
9,434,198,486


In [6]:
df[diagnosis_cols].dtypes

diag_1    object
diag_2    object
diag_3    object
dtype: object

In [7]:
df[diagnosis_cols].isna().sum()

diag_1      21
diag_2     358
diag_3    1423
dtype: int64

In [8]:
for col in diagnosis_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))


--- diag_1 ---
diag_1
428      6862
414      6581
786      4016
410      3614
486      3508
427      2766
491      2275
715      2151
682      2042
434      2028
780      2019
996      1967
276      1889
38       1688
250.8    1680
Name: count, dtype: int64

--- diag_2 ---
diag_2
276       6752
428       6662
250       6071
427       5036
401       3736
496       3305
599       3288
403       2823
414       2650
411       2566
250.02    2074
707       1999
585       1871
584       1649
491       1545
Name: count, dtype: int64

--- diag_3 ---
diag_3
250       11555
401        8289
276        5175
428        4577
427        3955
414        3664
496        2605
403        2357
585        1992
272        1969
599        1941
NaN        1423
V45        1389
250.02     1369
707        1360
Name: count, dtype: int64


In [9]:
def categorize_icd9(code):
    # Handle missing values
    if pd.isna(code):
        return 'Missing'

    code = str(code).strip().upper()

    # ICD-9 supplementary classification
    if code.startswith('V'):
        return 'Supplementary Factors'

    # ICD-9 external causes of injury
    if code.startswith('E'):
        return 'External Causes'

    # Extract the first three digits
    try:
        code_num = int(float(code))
    except (ValueError, TypeError):
        return 'Other/Unclassified'

    # ICD-9 major diagnostic categories
    if 1 <= code_num <= 139:
        return 'Infectious and Parasitic'

    elif 140 <= code_num <= 239:
        return 'Neoplasms'

    elif 240 <= code_num <= 279:
        return 'Endocrine, Metabolic and Nutritional'

    elif 280 <= code_num <= 289:
        return 'Blood and Blood-forming Organs'

    elif 290 <= code_num <= 319:
        return 'Mental Disorders'

    elif 320 <= code_num <= 359:
        return 'Nervous System'

    elif 360 <= code_num <= 389:
        return 'Sense Organs'

    elif 390 <= code_num <= 459:
        return 'Circulatory System'

    elif 460 <= code_num <= 519:
        return 'Respiratory System'

    elif 520 <= code_num <= 579:
        return 'Digestive System'

    elif 580 <= code_num <= 629:
        return 'Genitourinary System'

    elif 630 <= code_num <= 679:
        return 'Pregnancy and Childbirth'

    elif 680 <= code_num <= 709:
        return 'Skin and Subcutaneous Tissue'

    elif 710 <= code_num <= 739:
        return 'Musculoskeletal System'

    elif 740 <= code_num <= 759:
        return 'Congenital Conditions'

    elif 760 <= code_num <= 779:
        return 'Conditions Originating in Perinatal Period'

    elif 780 <= code_num <= 799:
        return 'Symptoms and Signs'

    elif 800 <= code_num <= 999:
        return 'Injury and Poisoning'

    else:
        return 'Other/Unclassified'

In [10]:
for col in diagnosis_cols:
    df[f'{col}_category'] = df[col].apply(categorize_icd9)

In [11]:
df[['diag_1', 'diag_1_category',
    'diag_2', 'diag_2_category',
    'diag_3', 'diag_3_category']].head(15)

,diag_1,diag_1_category,diag_2,diag_2_category,diag_3,diag_3_category
0,250.83,"Endocrine, Metabolic and Nutritional",NaN,Missing,NaN,Missing
1,276,"Endocrine, Metabolic and Nutritional",250.01,"Endocrine, Metabolic and Nutritional",255,"Endocrine, Metabolic and Nutritional"
2,648,Pregnancy and Childbirth,250,"Endocrine, Metabolic and Nutritional",V27,Supplementary Factors
3,8,Infectious and Parasitic,250.43,"Endocrine, Metabolic and Nutritional",403,Circulatory System
4,197,Neoplasms,157,Neoplasms,250,"Endocrine, Metabolic and Nutritional"
5,414,Circulatory System,411,Circulatory System,250,"Endocrine, Metabolic and Nutritional"
6,414,Circulatory System,411,Circulatory System,V45,Supplementary Factors
7,428,Circulatory System,492,Respiratory System,250,"Endocrine, Metabolic and Nutritional"
8,398,Circulatory System,427,Circulatory System,38,Infectious and Parasitic
9,434,Circulatory System,198,Neoplasms,486,Respiratory System


In [12]:
for col in diagnosis_cols:
    category_col = f'{col}_category'

    print(f"\n--- {category_col} ---")
    print(df[category_col].value_counts())


--- diag_1_category ---
diag_1_category
Circulatory System                      30336
Endocrine, Metabolic and Nutritional    11459
Respiratory System                      10407
Digestive System                         9208
Symptoms and Signs                       7636
Injury and Poisoning                     6974
Genitourinary System                     5078
Musculoskeletal System                   4957
Neoplasms                                3433
Infectious and Parasitic                 2768
Skin and Subcutaneous Tissue             2530
Mental Disorders                         2262
Supplementary Factors                    1644
Blood and Blood-forming Organs           1103
Nervous System                            947
Pregnancy and Childbirth                  687
Sense Organs                              264
Congenital Conditions                      51
Missing                                    21
External Causes                             1
Name: count, dtype: int64

--- diag_2_c

In [13]:
df.groupby('diag_1_category')['readmitted_30'].agg(
    encounters='count',
    readmission_rate='mean'
).sort_values('readmission_rate', ascending=False)

,encounters,readmission_rate
diag_1_category,,
Missing,21,0.238095
Supplementary Factors,1644,0.161800
Blood and Blood-forming Organs,1103,0.134180
"Endocrine, Metabolic and Nutritional",11459,0.127847
Injury and Poisoning,6974,0.122455
Mental Disorders,2262,0.122016
Nervous System,947,0.120380
Infectious and Parasitic,2768,0.114523
Circulatory System,30336,0.114517


In [14]:
result = (
    df.groupby('diag_1_category')['readmitted_30']
      .agg(encounters='count', readmission_rate='mean')
)

result['readmission_rate'] = result['readmission_rate'] * 100

result.sort_values('readmission_rate', ascending=False)

,encounters,readmission_rate
diag_1_category,,
Missing,21,23.809524
Supplementary Factors,1644,16.180049
Blood and Blood-forming Organs,1103,13.417951
"Endocrine, Metabolic and Nutritional",11459,12.784711
Injury and Poisoning,6974,12.245483
Mental Disorders,2262,12.201592
Nervous System,947,12.038015
Infectious and Parasitic,2768,11.452312
Circulatory System,30336,11.451741


In [15]:
diag1_distribution = (
    df['diag_1_category']
    .value_counts()
    .rename_axis('diag_1_category')
    .reset_index(name='encounters')
)

diag1_distribution['percentage'] = (
    diag1_distribution['encounters'] / len(df) * 100
)

diag1_distribution

,diag_1_category,encounters,percentage
0,Circulatory System,30336,29.809563
1,"Endocrine, Metabolic and Nutritional",11459,11.260146
2,Respiratory System,10407,10.226402
3,Digestive System,9208,9.048209
4,Symptoms and Signs,7636,7.503488
5,Injury and Poisoning,6974,6.852976
6,Genitourinary System,5078,4.989879
7,Musculoskeletal System,4957,4.870979
8,Neoplasms,3433,3.373425
9,Infectious and Parasitic,2768,2.719965


In [16]:
diag1_distribution.sort_values('encounters')

,diag_1_category,encounters,percentage
19,External Causes,1,0.000983
18,Missing,21,0.020636
17,Congenital Conditions,51,0.050115
16,Sense Organs,264,0.259419
15,Pregnancy and Childbirth,687,0.675078
14,Nervous System,947,0.930566
13,Blood and Blood-forming Organs,1103,1.083859
12,Supplementary Factors,1644,1.615471
11,Mental Disorders,2262,2.222746
10,Skin and Subcutaneous Tissue,2530,2.486096


In [17]:
diag1_distribution = (
    df['diag_1_category']
    .value_counts()
    .rename_axis('diag_1_category')
    .reset_index(name='encounters')
)

diag1_distribution['percentage'] = (
    diag1_distribution['encounters'] / len(df) * 100
)

diag1_distribution.sort_values('encounters')

,diag_1_category,encounters,percentage
19,External Causes,1,0.000983
18,Missing,21,0.020636
17,Congenital Conditions,51,0.050115
16,Sense Organs,264,0.259419
15,Pregnancy and Childbirth,687,0.675078
14,Nervous System,947,0.930566
13,Blood and Blood-forming Organs,1103,1.083859
12,Supplementary Factors,1644,1.615471
11,Mental Disorders,2262,2.222746
10,Skin and Subcutaneous Tissue,2530,2.486096


In [18]:
# Calculate category counts for the primary diagnosis
diag1_counts = df['diag_1_category'].value_counts()

# Identify categories representing at least 1% of encounters
threshold = len(df) * 0.01

common_diag1_categories = diag1_counts[
    diag1_counts >= threshold
].index

print("Total encounters:", len(df))
print("1% threshold:", round(threshold))
print("Common categories:", len(common_diag1_categories))
print("Rare categories:", len(diag1_counts) - len(common_diag1_categories))

print("\nCategories below the threshold:")
print(diag1_counts[diag1_counts < threshold])

Total encounters: 101766
1% threshold: 1018
Common categories: 14
Rare categories: 6

Categories below the threshold:
diag_1_category
Nervous System              947
Pregnancy and Childbirth    687
Sense Organs                264
Congenital Conditions        51
Missing                      21
External Causes               1
Name: count, dtype: int64


In [19]:
# Define a minimum count for this exploratory grouping
min_count = 500

# Count primary diagnosis categories
diag1_counts = df['diag_1_category'].value_counts()

# Identify rare categories, excluding Missing
rare_diag1_categories = diag1_counts[
    (diag1_counts < min_count) &
    (diag1_counts.index != 'Missing')
].index

# Create a temporary grouped feature
df['diag_1_category_grouped_temp'] = (
    df['diag_1_category']
    .where(
        ~df['diag_1_category'].isin(rare_diag1_categories),
        'Other/Uncommon'
    )
)

# Inspect the result
df['diag_1_category_grouped_temp'].value_counts()

diag_1_category_grouped_temp
Circulatory System                      30336
Endocrine, Metabolic and Nutritional    11459
Respiratory System                      10407
Digestive System                         9208
Symptoms and Signs                       7636
Injury and Poisoning                     6974
Genitourinary System                     5078
Musculoskeletal System                   4957
Neoplasms                                3433
Infectious and Parasitic                 2768
Skin and Subcutaneous Tissue             2530
Mental Disorders                         2262
Supplementary Factors                    1644
Blood and Blood-forming Organs           1103
Nervous System                            947
Pregnancy and Childbirth                  687
Other/Uncommon                            316
Missing                                    21
Name: count, dtype: int64

In [20]:
grouped_comparison = (
    df.groupby('diag_1_category_grouped_temp', observed=True)
      ['readmitted_30']
      .agg(
          encounters='count',
          readmissions='sum',
          readmission_rate='mean'
      )
)

grouped_comparison['readmission_rate'] *= 100

grouped_comparison.sort_values(
    'readmission_rate',
    ascending=False
)

,encounters,readmissions,readmission_rate
diag_1_category_grouped_temp,,,
Missing,21,5,23.809524
Supplementary Factors,1644,266,16.180049
Blood and Blood-forming Organs,1103,148,13.417951
"Endocrine, Metabolic and Nutritional",11459,1465,12.784711
Injury and Poisoning,6974,854,12.245483
Mental Disorders,2262,276,12.201592
Nervous System,947,114,12.038015
Infectious and Parasitic,2768,317,11.452312
Circulatory System,30336,3474,11.451741


In [21]:
# Group previous inpatient visits
df['inpatient_visits_group'] = pd.cut(
    df['number_inpatient'],
    bins=[-1, 0, 1, 2, 4, np.inf],
    labels=['0', '1', '2', '3-4', '5+']
)

# Group previous emergency visits
df['emergency_visits_group'] = pd.cut(
    df['number_emergency'],
    bins=[-1, 0, 1, 2, 4, np.inf],
    labels=['0', '1', '2', '3-4', '5+']
)

# Group previous outpatient visits
df['outpatient_visits_group'] = pd.cut(
    df['number_outpatient'],
    bins=[-1, 0, 1, 2, 4, 8, np.inf],
    labels=['0', '1', '2', '3-4', '5-8', '9+']
)

In [22]:
utilization_groups = [
    'inpatient_visits_group',
    'emergency_visits_group',
    'outpatient_visits_group'
]

for col in utilization_groups:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(sort=False, dropna=False))


--- inpatient_visits_group ---
inpatient_visits_group
0      67630
1      19521
2       7566
3-4     5033
5+      2016
Name: count, dtype: int64

--- emergency_visits_group ---
emergency_visits_group
0      90383
1       7677
2       2042
3-4     1099
5+       565
Name: count, dtype: int64

--- outpatient_visits_group ---
outpatient_visits_group
0      85027
1       8547
2       3594
3-4     3141
5-8     1089
9+       368
Name: count, dtype: int64


In [23]:
for col in utilization_groups:
    print(f"\n--- Readmission by {col} ---")

    result = (
        df.groupby(col, observed=False)['readmitted_30']
          .agg(
              encounters='count',
              readmission_rate='mean'
          )
    )

    result['readmission_rate'] *= 100

    print(result.round(2))


--- Readmission by inpatient_visits_group ---
                        encounters  readmission_rate
inpatient_visits_group                              
0                            67630              8.44
1                            19521             12.92
2                             7566             17.43
3-4                           5033             21.36
5+                            2016             36.41

--- Readmission by emergency_visits_group ---
                        encounters  readmission_rate
emergency_visits_group                              
0                            90383             10.47
1                             7677             14.35
2                             2042             18.27
3-4                           1099             23.84
5+                             565             27.08

--- Readmission by outpatient_visits_group ---
                         encounters  readmission_rate
outpatient_visits_group                              
0       

In [24]:
df['discharge_disposition'].value_counts(dropna=False)

discharge_disposition
Discharged to home                                                                                           60234
Discharged/transferred to SNF                                                                                13954
Discharged/transferred to home with home health service                                                      12902
NaN                                                                                                           3691
Discharged/transferred to another short term hospital                                                         2128
Discharged/transferred to another rehab fac including rehab units of a hospital .                             1993
Expired                                                                                                       1642
Discharged/transferred to another type of inpatient care institution                                          1184
Not Mapped                                                

In [25]:
discharge_summary = (
    df.groupby('discharge_disposition', dropna=False)
      ['readmitted_30']
      .agg(
          encounters='count',
          readmissions='sum',
          readmission_rate='mean'
      )
)

discharge_summary['readmission_rate'] *= 100

discharge_summary.sort_values(
    'encounters',
    ascending=False
).round(2)

,encounters,readmissions,readmission_rate
discharge_disposition,,,
Discharged to home,60234,5602,9.30
Discharged/transferred to SNF,13954,2046,14.66
Discharged/transferred to home with home health service,12902,1638,12.70
NaN,3691,459,12.44
Discharged/transferred to another short term hospital,2128,342,16.07
Discharged/transferred to another rehab fac including rehab units of a hospital .,1993,552,27.70
Expired,1642,0,0.00
Discharged/transferred to another type of inpatient care institution,1184,247,20.86
Not Mapped,989,92,9.30


In [26]:
discharge_summary['percentage'] = (
    discharge_summary['encounters'] / len(df) * 100
)

discharge_summary.sort_values(
    'encounters',
    ascending=False
).round(2)

,encounters,readmissions,readmission_rate,percentage
discharge_disposition,,,,
Discharged to home,60234,5602,9.30,59.19
Discharged/transferred to SNF,13954,2046,14.66,13.71
Discharged/transferred to home with home health service,12902,1638,12.70,12.68
NaN,3691,459,12.44,3.63
Discharged/transferred to another short term hospital,2128,342,16.07,2.09
Discharged/transferred to another rehab fac including rehab units of a hospital .,1993,552,27.70,1.96
Expired,1642,0,0.00,1.61
Discharged/transferred to another type of inpatient care institution,1184,247,20.86,1.16
Not Mapped,989,92,9.30,0.97


In [27]:
df['discharge_disposition'].value_counts(dropna=False).index.tolist()

['Discharged to home',
 'Discharged/transferred to SNF',
 'Discharged/transferred to home with home health service',
 nan,
 'Discharged/transferred to another short term hospital',
 'Discharged/transferred to another rehab fac including rehab units of a hospital .',
 'Expired',
 'Discharged/transferred to another type of inpatient care institution',
 'Not Mapped',
 'Discharged/transferred to ICF',
 'Left AMA',
 'Discharged/transferred to a long term care hospital.',
 'Hospice / home',
 'Hospice / medical facility',
 'Discharged/transferred/referred to a psychiatric hospital of psychiatric distinct part unit of a hospital',
 'Discharged/transferred to home under care of Home IV provider',
 'Discharged/transferred within this institution to Medicare approved swing bed',
 'Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.',
 'Admitted as an inpatient to this hospital',
 'Discharged/transferred/referred to this institution for outpatien

In [28]:
def group_discharge_disposition(value):

    # Handle missing and unmapped values separately
    if pd.isna(value):
        return 'Missing'

    if value == 'Not Mapped':
        return 'Not Mapped'

    # Death-related outcomes
    if value in [
        'Expired',
        'Expired at home. Medicaid only, hospice.',
        'Expired in a medical facility. Medicaid only, hospice.'
    ]:
        return 'Expired'

    # Hospice
    if value in [
        'Hospice / home',
        'Hospice / medical facility'
    ]:
        return 'Hospice'

    # Home discharge
    if value == 'Discharged to home':
        return 'Home'

    # Home with additional care
    if value in [
        'Discharged/transferred to home with home health service',
        'Discharged/transferred to home under care of Home IV provider'
    ]:
        return 'Home with Support Services'

    # Skilled nursing facility
    if value in [
        'Discharged/transferred to SNF',
        'Discharged/transferred to a nursing facility certified under Medicaid but not certified under Medicare.'
    ]:
        return 'Skilled Nursing Facility'

    # Rehabilitation
    if value == (
        'Discharged/transferred to another rehab fac '
        'including rehab units of a hospital .'
    ):
        return 'Rehabilitation Facility'

    # Left against medical advice
    if value == 'Left AMA':
        return 'Left Against Medical Advice'

    # Other transfers and discharge destinations
    return 'Other Facility/Transfer'
    

In [29]:
df['discharge_group'] = (
    df['discharge_disposition']
    .apply(group_discharge_disposition)
)

In [30]:
df['discharge_group'].value_counts(dropna=False)

discharge_group
Home                           60234
Skilled Nursing Facility       14002
Home with Support Services     13010
Other Facility/Transfer         4801
Missing                         3691
Rehabilitation Facility         1993
Expired                         1652
Not Mapped                       989
Hospice                          771
Left Against Medical Advice      623
Name: count, dtype: int64

In [31]:
discharge_group_summary = (
    df.groupby('discharge_group', observed=True)['readmitted_30']
      .agg(
          encounters='count',
          readmissions='sum',
          readmission_rate='mean'
      )
)

discharge_group_summary['readmission_rate'] *= 100

discharge_group_summary.sort_values(
    'encounters',
    ascending=False
).round(2)

,encounters,readmissions,readmission_rate
discharge_group,,,
Home,60234,5602,9.30
Skilled Nursing Facility,14002,2053,14.66
Home with Support Services,13010,1653,12.71
Other Facility/Transfer,4801,813,16.93
Missing,3691,459,12.44
Rehabilitation Facility,1993,552,27.70
Expired,1652,0,0.00
Not Mapped,989,92,9.30
Hospice,771,43,5.58


In [32]:
print("Original categories:", df['discharge_disposition'].nunique(dropna=False))
print("New groups:", df['discharge_group'].nunique(dropna=False))

print("\nMissing original values:")
print(df['discharge_disposition'].isna().sum())

print("\nMissing grouped values:")
print(df['discharge_group'].isna().sum())

Original categories: 26
New groups: 10

Missing original values:
3691

Missing grouped values:
0


In [33]:
complexity_cols = [
    'number_diagnoses',
    'num_medications',
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures'
]

df[complexity_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
number_diagnoses,101766.0,7.422607,1.933600,1.0,6.0,8.0,9.0,16.0
num_medications,101766.0,16.021844,8.127566,1.0,10.0,15.0,20.0,81.0
time_in_hospital,101766.0,4.395987,2.985108,1.0,2.0,4.0,6.0,14.0
num_lab_procedures,101766.0,43.095641,19.674362,1.0,31.0,44.0,57.0,132.0
num_procedures,101766.0,1.339730,1.705807,0.0,0.0,1.0,2.0,6.0


In [34]:
df[complexity_cols].nunique()

number_diagnoses       16
num_medications        75
time_in_hospital       14
num_lab_procedures    118
num_procedures          7
dtype: int64

In [35]:
for col in complexity_cols:
    print(f"\n--- Readmission by {col} ---")

    summary = (
        df.groupby(col, observed=True)['readmitted_30']
          .agg(
              encounters='count',
              readmission_rate='mean'
          )
    )

    summary['readmission_rate'] *= 100

    print(summary.round(2))


--- Readmission by number_diagnoses ---
                  encounters  readmission_rate
number_diagnoses                              
1                        219              5.94
2                       1023              6.06
3                       2835              7.37
4                       5537              8.25
5                      11393              9.15
6                      10161             10.41
7                      10393             10.77
8                      10616             11.81
9                      49474             12.38
10                        17             17.65
11                        11             27.27
12                         9             11.11
13                        16             18.75
14                         7             14.29
15                        10             20.00
16                        45              8.89

--- Readmission by num_medications ---
                 encounters  readmission_rate
num_medications            

In [36]:
# Inspect HbA1c result categories
print(df['A1Cresult'].value_counts(dropna=False))

print("\nReadmission by A1Cresult:")
print(
    df.groupby('A1Cresult', dropna=False)['readmitted_30']
      .agg(
          encounters='count',
          readmission_rate='mean'
      )
      .assign(
          readmission_rate=lambda x: x['readmission_rate'] * 100
      )
      .round(2)
)

A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64

Readmission by A1Cresult:
           encounters  readmission_rate
A1Cresult                              
>7               3812             10.05
>8               8216              9.87
Norm             4990              9.66
NaN             84748             11.42


In [37]:
# Inspect diabetes medication status
print(df['diabetesMed'].value_counts(dropna=False))

print("\nReadmission by diabetesMed:")
print(
    df.groupby('diabetesMed', dropna=False)['readmitted_30']
      .agg(
          encounters='count',
          readmission_rate='mean'
      )
      .assign(
          readmission_rate=lambda x: x['readmission_rate'] * 100
      )
      .round(2)
)

diabetesMed
Yes    78363
No     23403
Name: count, dtype: int64

Readmission by diabetesMed:
             encounters  readmission_rate
diabetesMed                              
No                23403              9.60
Yes               78363             11.63


In [38]:
# 1 = A1C result is missing
# 0 = A1C result is available

df['A1C_missing'] = df['A1Cresult'].isna().astype(int)

In [39]:
print(df['A1C_missing'].value_counts())

print("\nReadmission by A1C missingness:")

a1c_missing_summary = (
    df.groupby('A1C_missing')['readmitted_30']
      .agg(
          encounters='count',
          readmission_rate='mean'
      )
)

a1c_missing_summary['readmission_rate'] *= 100

a1c_missing_summary.round(2)

A1C_missing
1    84748
0    17018
Name: count, dtype: int64

Readmission by A1C missingness:


,encounters,readmission_rate
A1C_missing,,
0,17018,9.85
1,84748,11.42


In [40]:
# Inspect all three diagnosis columns together
diagnosis_cols = ['diag_1', 'diag_2', 'diag_3']

all_diagnosis_codes = df[diagnosis_cols].stack()

# Count the different code formats
print("Total non-missing diagnosis values:", len(all_diagnosis_codes))

print("\nCodes starting with V:")
print(
    all_diagnosis_codes[
        all_diagnosis_codes.astype(str).str.startswith('V', na=False)
    ].value_counts().head(15)
)

print("\nCodes starting with E:")
print(
    all_diagnosis_codes[
        all_diagnosis_codes.astype(str).str.startswith('E', na=False)
    ].value_counts().head(15)
)

print("\nMost frequent codes containing decimals:")
print(
    all_diagnosis_codes[
        all_diagnosis_codes.astype(str).str.contains('.', regex=False, na=False)
    ].value_counts().head(15)
)

Total non-missing diagnosis values: 303496

Codes starting with V:
V45    1802
V57    1223
V58     886
V42     507
V15     462
V43     342
V10     340
V12     271
V85     265
V54     181
V09      96
V62      95
V49      93
V55      81
V64      72
Name: count, dtype: int64

Codes starting with E:
E878    277
E849    273
E888    240
E885    229
E932    109
E942     86
E879     79
E947     59
E935     54
E884     45
E880     45
E939     33
E934     32
E944     31
E812     29
Name: count, dtype: int64

Most frequent codes containing decimals:
250.02    4118
250.6     3158
250.01    2499
250.8     2201
250.7     1142
250.13     967
250.4      866
250.11     722
250.82     702
250.03     568
250.12     552
250.41     518
250.92     402
250.1      350
250.81     330
Name: count, dtype: int64


In [41]:
import re

def check_diagnosis_code(code):
    if pd.isna(code):
        return "Missing"

    code = str(code).strip().upper()

    if re.fullmatch(r"\d{3}(?:\.\d+)?", code):
        return "Valid numeric format"
    elif re.fullmatch(r"V\d{2,3}(?:\.\d+)?", code):
        return "V-code format"
    elif re.fullmatch(r"E\d{3,4}", code):
        return "E-code format"
    else:
        return "Unexpected format"


for col in diagnosis_cols:
    print(f"\n{col}")
    print(df[col].apply(check_diagnosis_code).value_counts())


diag_1
diag_1
Valid numeric format    97454
Unexpected format        2646
V-code format            1644
Missing                    21
E-code format               1
Name: count, dtype: int64

diag_2
diag_2
Valid numeric format    97312
V-code format            1805
Unexpected format        1560
E-code format             731
Missing                   358
Name: count, dtype: int64

diag_3
diag_3
Valid numeric format    93759
V-code format            3814
Unexpected format        1526
Missing                  1423
E-code format            1244
Name: count, dtype: int64


In [42]:
for col in diagnosis_cols:
    unexpected_mask = (
        df[col].notna()
        & df[col].apply(check_diagnosis_code).eq("Unexpected format")
    )

    print(f"\n--- {col}: Unexpected values ---")
    print(df.loc[unexpected_mask, col].value_counts().head(20))


--- diag_1: Unexpected values ---
diag_1
38    1688
8      515
79      86
53      55
70      46
9       46
42      45
47      26
3       13
54      12
11      10
35      10
78       9
88       9
5        9
41       8
82       7
34       7
94       7
66       6
Name: count, dtype: int64

--- diag_2: Unexpected values ---
diag_2
38    475
41    379
70    273
8     199
42     85
53     32
79     29
94     26
40     12
34     10
54     10
9       7
31      4
11      3
27      3
78      2
75      2
5       1
35      1
46      1
Name: count, dtype: int64

--- diag_3: Unexpected values ---
diag_3
41    727
70    281
8     190
38    162
79     46
53     25
42     23
94     21
54     14
34      6
9       5
35      4
5       3
78      3
3       2
11      2
7       2
88      2
75      1
27      1
Name: count, dtype: int64


In [43]:
for col in diagnosis_cols:
    unexpected_mask = (
        df[col].notna()
        & df[col].apply(check_diagnosis_code).eq("Unexpected format")
    )

    print(f"\n--- {col}: Python data types ---")
    print(df.loc[unexpected_mask, col].map(type).value_counts())

    print(f"\n--- {col}: Sample values and representations ---")
    for value in df.loc[unexpected_mask, col].drop_duplicates().head(10):
        print(repr(value))


--- diag_1: Python data types ---
diag_1
<class 'str'>    2646
Name: count, dtype: int64

--- diag_1: Sample values and representations ---
'8'
'38'
'70'
'82'
'54'
'78'
'79'
'41'
'3'
'53'

--- diag_2: Python data types ---
diag_2
<class 'str'>    1560
Name: count, dtype: int64

--- diag_2: Sample values and representations ---
'41'
'70'
'53'
'8'
'38'
'34'
'78'
'79'
'42'
'54'

--- diag_3: Python data types ---
diag_3
<class 'str'>    1526
Name: count, dtype: int64

--- diag_3: Sample values and representations ---
'38'
'8'
'41'
'54'
'75'
'70'
'42'
'35'
'94'
'79'


In [44]:
# Load the original dataset for comparison
raw_df = pd.read_csv("../data/raw/diabetic_data.csv", dtype=str)

for col in diagnosis_cols:
    print(f"\n--- {col}: Raw dataset ---")
    print(raw_df[col].value_counts().head(10))

    print(f"\n--- {col}: Cleaned dataset ---")
    print(df[col].value_counts().head(10))


--- diag_1: Raw dataset ---
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64

--- diag_1: Cleaned dataset ---
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64

--- diag_2: Raw dataset ---
diag_2
276    6752
428    6662
250    6071
427    5036
401    3736
496    3305
599    3288
403    2823
414    2650
411    2566
Name: count, dtype: int64

--- diag_2: Cleaned dataset ---
diag_2
276    6752
428    6662
250    6071
427    5036
401    3736
496    3305
599    3288
403    2823
414    2650
411    2566
Name: count, dtype: int64

--- diag_3: Raw dataset ---
diag_3
250    11555
401     8289
276     5175
428     4577
427     3955
414     3664
496     2605
403     2357
585     1992
272     1969
Name: count, dtype: int64

--- diag_3: Cleaned dataset ---
diag_3
250    11555
401     8289
276   

In [45]:
import re

def check_diagnosis_code(code):
    if pd.isna(code):
        return "Missing"

    code = str(code).strip().upper()

    if re.fullmatch(r"\d{1,3}(?:\.\d+)?", code):
        return "Numeric format"
    elif re.fullmatch(r"V\d{2,3}(?:\.\d+)?", code):
        return "V-code format"
    elif re.fullmatch(r"E\d{3,4}", code):
        return "E-code format"
    else:
        return "Unexpected format"


for col in diagnosis_cols:
    print(f"\n{col}")
    print(df[col].apply(check_diagnosis_code).value_counts())


diag_1
diag_1
Numeric format    100100
V-code format       1644
Missing               21
E-code format          1
Name: count, dtype: int64

diag_2
diag_2
Numeric format    98872
V-code format      1805
E-code format       731
Missing             358
Name: count, dtype: int64

diag_3
diag_3
Numeric format    95285
V-code format      3814
Missing            1423
E-code format      1244
Name: count, dtype: int64


In [46]:
def categorize_icd9(code):
    if pd.isna(code):
        return "Missing"

    code = str(code).strip().upper()

    # Handle supplementary and external-cause codes first
    if code.startswith("V"):
        return "Supplementary Factors"

    if code.startswith("E"):
        return "External Causes"

    # Extract the numeric category before the decimal
    match = re.fullmatch(r"(\d{1,3})(?:\.\d+)?", code)

    if not match:
        return "Other/Unclassified"

    code_num = int(match.group(1))

    if 1 <= code_num <= 139:
        return "Infectious and Parasitic"
    elif 140 <= code_num <= 239:
        return "Neoplasms"
    elif 240 <= code_num <= 279:
        return "Endocrine, Metabolic and Nutritional"
    elif 280 <= code_num <= 289:
        return "Blood and Blood-forming Organs"
    elif 290 <= code_num <= 319:
        return "Mental Disorders"
    elif 320 <= code_num <= 359:
        return "Nervous System"
    elif 360 <= code_num <= 389:
        return "Sense Organs"
    elif 390 <= code_num <= 459:
        return "Circulatory System"
    elif 460 <= code_num <= 519:
        return "Respiratory System"
    elif 520 <= code_num <= 579:
        return "Digestive System"
    elif 580 <= code_num <= 629:
        return "Genitourinary System"
    elif 630 <= code_num <= 679:
        return "Pregnancy and Childbirth"
    elif 680 <= code_num <= 709:
        return "Skin and Subcutaneous Tissue"
    elif 710 <= code_num <= 739:
        return "Musculoskeletal System"
    elif 740 <= code_num <= 759:
        return "Congenital Conditions"
    elif 760 <= code_num <= 779:
        return "Conditions Originating in Perinatal Period"
    elif 780 <= code_num <= 799:
        return "Symptoms and Signs"
    elif 800 <= code_num <= 999:
        return "Injury and Poisoning"
    else:
        return "Other/Unclassified"

In [47]:
for col in diagnosis_cols:
    df[f"{col}_category"] = df[col].apply(categorize_icd9)

In [48]:
for col in diagnosis_cols:
    category_col = f"{col}_category"

    print(f"\n--- {category_col}: Category counts ---")
    print(df[category_col].value_counts(dropna=False))

    print("\nUnclassified count:")
    print(df[category_col].eq("Other/Unclassified").sum())


--- diag_1_category: Category counts ---
diag_1_category
Circulatory System                      30336
Endocrine, Metabolic and Nutritional    11459
Respiratory System                      10407
Digestive System                         9208
Symptoms and Signs                       7636
Injury and Poisoning                     6974
Genitourinary System                     5078
Musculoskeletal System                   4957
Neoplasms                                3433
Infectious and Parasitic                 2768
Skin and Subcutaneous Tissue             2530
Mental Disorders                         2262
Supplementary Factors                    1644
Blood and Blood-forming Organs           1103
Nervous System                            947
Pregnancy and Childbirth                  687
Sense Organs                              264
Congenital Conditions                      51
Missing                                    21
External Causes                             1
Name: count, dtype: in

In [49]:
diag1_counts = df["diag_1_category"].value_counts()

print("Categories with fewer than 500 encounters:")
print(diag1_counts[diag1_counts < 500])

print("\nNumber of categories:", diag1_counts.size)

Categories with fewer than 500 encounters:
diag_1_category
Sense Organs             264
Congenital Conditions     51
Missing                   21
External Causes            1
Name: count, dtype: int64

Number of categories: 20


In [50]:
for col in [
    "diag_1_category",
    "diag_2_category",
    "diag_3_category"
]:
    print(f"\n--- {col}: Categories with fewer than 500 encounters ---")

    counts = df[col].value_counts()
    print(counts[counts < 500])


--- diag_1_category: Categories with fewer than 500 encounters ---
diag_1_category
Sense Organs             264
Congenital Conditions     51
Missing                   21
External Causes            1
Name: count, dtype: int64

--- diag_2_category: Categories with fewer than 500 encounters ---
diag_2_category
Pregnancy and Childbirth    415
Missing                     358
Sense Organs                160
Congenital Conditions       108
Name: count, dtype: int64

--- diag_3_category: Categories with fewer than 500 encounters ---
diag_3_category
Pregnancy and Childbirth    309
Sense Organs                263
Congenital Conditions        96
Name: count, dtype: int64


In [51]:
print("Dataset shape:", df.shape)

print("\nEngineered columns:")
print([
    col for col in df.columns
    if col.endswith("_category")
    or col in ["discharge_group", "A1C_missing"]
])

print("\nTarget column:")
print(df["readmitted_30"].value_counts(dropna=False))

Dataset shape: (101766, 59)

Engineered columns:
['diag_1_category', 'diag_2_category', 'diag_3_category', 'discharge_group', 'A1C_missing']

Target column:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64


In [52]:
# Compare A1C missingness with the engineered indicator
actual_missing = df["A1Cresult"].isna().astype(int)

print("Does A1C_missing exactly match A1Cresult missingness?")
print(df["A1C_missing"].equals(actual_missing))

print("\nA1Cresult values:")
print(df["A1Cresult"].value_counts(dropna=False))

print("\nA1C_missing values:")
print(df["A1C_missing"].value_counts(dropna=False))

Does A1C_missing exactly match A1Cresult missingness?
True

A1Cresult values:
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64

A1C_missing values:
A1C_missing
1    84748
0    17018
Name: count, dtype: int64


In [53]:
candidate_cols = [
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
    "discharge_group",
    "A1Cresult",
    "diabetesMed",
    "number_inpatient",
    "number_emergency",
    "number_outpatient",
]

missing_summary = pd.DataFrame({
    "missing_count": df[candidate_cols].isna().sum(),
    "missing_percent": (df[candidate_cols].isna().mean() * 100).round(2)
})

print(missing_summary.sort_values("missing_count", ascending=False))

                   missing_count  missing_percent
A1Cresult                  84748            83.28
diag_1_category                0             0.00
diag_2_category                0             0.00
diag_3_category                0             0.00
discharge_group                0             0.00
diabetesMed                    0             0.00
number_inpatient               0             0.00
number_emergency               0             0.00
number_outpatient              0             0.00


In [54]:
print("All columns in the current dataset:\n")

for i, col in enumerate(df.columns, start=1):
    print(f"{i:02d}. {col}")

All columns in the current dataset:

01. encounter_id
02. patient_nbr
03. race
04. gender
05. age
06. admission_type_id
07. discharge_disposition_id
08. admission_source_id
09. time_in_hospital
10. payer_code
11. medical_specialty
12. num_lab_procedures
13. num_procedures
14. num_medications
15. number_outpatient
16. number_emergency
17. number_inpatient
18. diag_1
19. diag_2
20. diag_3
21. number_diagnoses
22. A1Cresult
23. metformin
24. repaglinide
25. nateglinide
26. chlorpropamide
27. glimepiride
28. acetohexamide
29. glipizide
30. glyburide
31. tolbutamide
32. pioglitazone
33. rosiglitazone
34. acarbose
35. miglitol
36. troglitazone
37. tolazamide
38. insulin
39. glyburide-metformin
40. glipizide-metformin
41. glimepiride-pioglitazone
42. metformin-rosiglitazone
43. metformin-pioglitazone
44. change
45. diabetesMed
46. readmitted
47. admission_type
48. discharge_disposition
49. admission_source
50. readmitted_30
51. diag_1_category
52. diag_2_category
53. diag_3_category
54. diag_

In [55]:
utilization_pairs = {
    "number_inpatient": "inpatient_visits_group",
    "number_emergency": "emergency_visits_group",
    "number_outpatient": "outpatient_visits_group",
}

for original, grouped in utilization_pairs.items():
    print(f"\n--- {original} vs {grouped} ---")

    print("Grouped feature counts:")
    print(df[grouped].value_counts(dropna=False))

    print("\nOriginal values by group:")
    print(
        df.groupby(grouped, observed=False)[original]
        .agg(["min", "max", "count"])
    )


--- number_inpatient vs inpatient_visits_group ---
Grouped feature counts:
inpatient_visits_group
0      67630
1      19521
2       7566
3-4     5033
5+      2016
Name: count, dtype: int64

Original values by group:
                        min  max  count
inpatient_visits_group                 
0                         0    0  67630
1                         1    1  19521
2                         2    2   7566
3-4                       3    4   5033
5+                        5   21   2016

--- number_emergency vs emergency_visits_group ---
Grouped feature counts:
emergency_visits_group
0      90383
1       7677
2       2042
3-4     1099
5+       565
Name: count, dtype: int64

Original values by group:
                        min  max  count
emergency_visits_group                 
0                         0    0  90383
1                         1    1   7677
2                         2    2   2042
3-4                       3    4   1099
5+                        5   76    565

--- n

In [56]:
print(
    "A1C_missing matches A1Cresult missingness:",
    df["A1C_missing"].equals(df["A1Cresult"].isna().astype(int))
)

print(
    "\nTemporary grouped diagnosis column exists:",
    "diag_1_category_grouped_temp" in df.columns
)

print(
    "\nTarget has missing values:",
    df["readmitted_30"].isna().sum()
)

A1C_missing matches A1Cresult missingness: True

Temporary grouped diagnosis column exists: True

Target has missing values: 0


In [57]:
# Remove temporary and redundant engineered features
df = df.drop(
    columns=["diag_1_category_grouped_temp", "A1C_missing"]
)

print("Updated dataset shape:", df.shape)

print("\nColumns removed:")
print(["diag_1_category_grouped_temp", "A1C_missing"])

Updated dataset shape: (101766, 57)

Columns removed:
['diag_1_category_grouped_temp', 'A1C_missing']


In [58]:
engineered_cols = [
    "diag_1_category",
    "diag_2_category",
    "diag_3_category",
    "discharge_group",
    "inpatient_visits_group",
    "emergency_visits_group",
    "outpatient_visits_group",
]

print("Engineered feature check:")
for col in engineered_cols:
    print(f"{col}: {'Present' if col in df.columns else 'Missing'}")

print("\nDataset shape:", df.shape)
print("\nTarget distribution:")
print(df["readmitted_30"].value_counts(dropna=False))

Engineered feature check:
diag_1_category: Present
diag_2_category: Present
diag_3_category: Present
discharge_group: Present
inpatient_visits_group: Present
emergency_visits_group: Present
outpatient_visits_group: Present

Dataset shape: (101766, 57)

Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64


In [59]:
# Final data-quality checks
print("Dataset shape:", df.shape)

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMissing target values:", df["readmitted_30"].isna().sum())

print("\nMissing values in engineered features:")
print(df[engineered_cols].isna().sum())

print("\nTarget distribution:")
print(df["readmitted_30"].value_counts())

Dataset shape: (101766, 57)

Duplicate rows: 0

Missing target values: 0

Missing values in engineered features:
diag_1_category            0
diag_2_category            0
diag_3_category            0
discharge_group            0
inpatient_visits_group     0
emergency_visits_group     0
outpatient_visits_group    0
dtype: int64

Target distribution:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64


In [60]:
# Save the feature-engineered dataset
output_path = "../data/processed/feature_engineered_diabetes_data.csv"

df.to_csv(output_path, index=False)

print("Feature-engineered dataset saved successfully.")
print("File:", output_path)
print("Shape:", df.shape)

PermissionError: [Errno 13] Permission denied: '../data/processed/feature_engineered_diabetes_data.csv'

In [61]:
df_check = pd.read_csv(output_path, low_memory=False)

print("Reloaded shape:", df_check.shape)
print("\nTarget distribution after reload:")
print(df_check["readmitted_30"].value_counts())

Reloaded shape: (101766, 57)

Target distribution after reload:
readmitted_30
0    90409
1    11357
Name: count, dtype: int64
